In [1]:
import serial
import serial.tools.list_ports
import time
import numpy as np

# --- Konfiguration ---
VIDPID   = "16C0:0483"   # Teensy 4.1 VID:PID
BAUDRATE = 9600
TIMEOUT  = 5

BLOCK_SIZE   = 1500
N_SAMPLES    = 10_000_000
ADC_CHANNEL  = 0
SYNC_BYTE    = 0x7E

def find_port(vidpid=VIDPID):
    for p in serial.tools.list_ports.comports():
        if vidpid in p.hwid:
            return p.device
    raise IOError("Teensy ikke fundet (VID:PID matcher ikke).")

def open_serial():
    port = find_port()
    ser = serial.Serial(port, baudrate=BAUDRATE, timeout=TIMEOUT)
    ser.dtr = False
    time.sleep(0.05)
    ser.reset_input_buffer()
    ser.reset_output_buffer()
    return ser

def sync(ser, overall=5.0):
    """Find næste sync-byte 0x7E i strømmen."""
    start = time.monotonic()
    while True:
        if time.monotonic() - start > overall:
            raise RuntimeError("SYNC_TIMEOUT (fandt ikke 0x7E)")
        b = ser.read(1)
        if not b:
            continue
        if b[0] == SYNC_BYTE:
            return

def read_exact(ser, n):
    """Læs præcis n bytes eller fejler."""
    buf = bytearray()
    while len(buf) < n:
        chunk = ser.read(n - len(buf))
        if not chunk:
            break
        buf.extend(chunk)
    if len(buf) != n:
        raise RuntimeError(f"Timeout på data: manglede {n - len(buf)} bytes")
    return bytes(buf)

def read_block(ser):
    """
    Læs én blok i det nye format:
      0x7E
      adc_num (1 byte)
      count   (2 byte, big-endian)
      block_id (4 byte, big-endian)
      data    (count * uint16)
    """
    sync(ser)

    # Ny header: 7 bytes efter sync
    hdr = read_exact(ser, 7)

    adc = hdr[0]
    count = (hdr[1] << 8) | hdr[2]
    block_id = (
        (hdr[3] << 24) |
        (hdr[4] << 16) |
        (hdr[5] << 8)  |
        hdr[6]
    )

    need = count * 2
    data = read_exact(ser, need)

    # Teensy sender uint16 råt fra RAM.
    # På Teensy 4.1 er det little-endian.
    block = np.frombuffer(data, dtype='<u2')

    return adc, count, block_id, block

def main():
    ser = open_serial()
    print(f"Forbundet til {ser.port}")

    # beregn antal blokke
    blocks_needed = (N_SAMPLES + BLOCK_SIZE - 1) // BLOCK_SIZE

    # send START
    ser.write(f"START {blocks_needed}\n".encode())
    ser.flush()
    time.sleep(0.12)

    zeros = 0
    ones = 0
    got = 0
    last_bid = None

    try:
        while got < N_SAMPLES:
            adc, count, block_id, block = read_block(ser)

            # valgfri kontrol
            if count != len(block):
                raise RuntimeError(
                    f"Count mismatch: header={count}, faktisk={len(block)}"
                )

            # check for mistede blokke på samme kanal
            if adc == ADC_CHANNEL:
                if last_bid is not None and block_id != last_bid + 1:
                    print(
                        f"Advarsel: block_id hop på adc {adc}: "
                        f"{last_bid} -> {block_id}"
                    )
                last_bid = block_id

            if adc != ADC_CHANNEL:
                continue

            need = N_SAMPLES - got
            raw = block[:need] if len(block) > need else block

            lsb = raw & 1
            ones_in_block = int(lsb.sum())
            zeros_in_block = int(len(lsb) - ones_in_block)

            ones += ones_in_block
            zeros += zeros_in_block
            got += int(len(raw))

            if got % 50000 == 0:
                print(f"{got}/{N_SAMPLES} samples... seneste block_id={block_id}")

    finally:
        try:
            ser.write(b"STOP\n")
            ser.flush()
            time.sleep(0.05)
        except Exception:
            pass
        ser.close()

    print("\n=== RESULTAT (raw samples) ===")
    print(f"Samples i alt : {got}")
    print(f"LSB = 0       : {zeros}")
    print(f"LSB = 1       : {ones}")

    if zeros > 0:
        ratio = ones / zeros
        print(f"Forhold (1/0) : {ratio:.6f}")

    frac_ones = ones / got if got else float("nan")
    bias = frac_ones - 0.5
    print(f"Andel 1'ere   : {frac_ones:.6f}")
    print(f"Bias (p-0.5)  : {bias:.6f}")

if __name__ == "__main__":
    main()

Forbundet til COM3
150000/10000000 samples... seneste block_id=99
300000/10000000 samples... seneste block_id=199
450000/10000000 samples... seneste block_id=299
600000/10000000 samples... seneste block_id=399
750000/10000000 samples... seneste block_id=499
900000/10000000 samples... seneste block_id=599
1050000/10000000 samples... seneste block_id=699
1200000/10000000 samples... seneste block_id=799
1350000/10000000 samples... seneste block_id=899
1500000/10000000 samples... seneste block_id=999
1650000/10000000 samples... seneste block_id=1099
1800000/10000000 samples... seneste block_id=1199
1950000/10000000 samples... seneste block_id=1299
2100000/10000000 samples... seneste block_id=1399
2250000/10000000 samples... seneste block_id=1499
2400000/10000000 samples... seneste block_id=1599
2550000/10000000 samples... seneste block_id=1699
2700000/10000000 samples... seneste block_id=1799
2850000/10000000 samples... seneste block_id=1899
3000000/10000000 samples... seneste block_id=199